# Ground truth on the probe stacks — compute

Writes the per-link files `gt_analysis.ipynb` reads (the port of `epanet_gt_V4`'s sanity tests onto the pipeline's exact
layer). Reads nothing but the exact layer of each stack (`fedwater.placement.exact`, cached per stack physics), the
world's `sensor_placement.csv` for the placed sensors, and — when `dose_response.ipynb` ran on the same exact layer —
its dose runs. Every table is restricted to the **placement candidates** (`placement.select.candidates`: home district
known, not a pump / valve / dead link, mean |q| ≥ `flow_floor` × the network median); maps draw every link.

**Stacks.** The LABEL stack of one built world per (network, partition): the probe stack the placed sensors' labels
come from (seed of the probes, K worlds, world k converts district k). The exact layer needs `e2_traj`, `periods` and
`cards` (all on by default since `EXACT_VERSION` 2): 1 no-drift run, K + 1 twins on the no-drift course, and per k the
drift world re-simulated plus two twins on its course — 4K + 2 full-horizon runs per stack, cached.

| file | one row per | content |
|---|---|---|
| `labels` | link × k × period | g_w, N = \|g_w\|, S, N_r per period (init = the stack's baseline window, transition, final = k's settled window); the final window's exact quantities on every row: Δm_k, m_k, σ, interaction, g_de0/1, g_ie0/1, S_de0, classes, channels, γ |
| `monthly`, `pumps_monthly` | link × k × month / pump × k × month | g_w, S, N_r (same month), N_r against the own final response; pumps' on-fraction with and without the drift |
| `composition` | link × world × window | raw destination shares (districts, filling tanks, reservoir, unaccounted): no-drift city on the baseline window; no-drift city and drift world on each k's settled window |
| `matrix`, `matrix_topo` | partition × client j × k × metric × scope | share of j's links responding to k, median label |
| `sensitivity` | partition × metric × floor | responding shares, district-blind share, branch false responses |
| `relations`, `m3`, `stationarity`, `channel` | partition / partition × response / link × k window / link × k × metric | how the labels relate; composition as a predictor; window dependence of the composition; exact channel |
| `links`, `geometry`, `nodes`, `periods`, `gates` | link / link / node / k / gate | link table with placement; map geometry; windows in hours; exact (G-X…) and dose (G-D…) gates |
| `cards/<network>/<partition>/<k>.npz`, `…/dose/*.npz` | k / k × λ | the exact layer's and the dose runs' card series (hard links) |

### Definitions

With $\Delta Q_s=Q^{(k)}_s-Q^{(0)}_s$ against the spliced no-drift city and $P$ the weekly profile, on a window $W$:

| quantity | definition | reads as |
|---|---|---|
| $g_w$ | $\langle P\Delta Q_s,P\Delta D_k\rangle/\lVert P\Delta D_k\rVert^2$, oriented by the no-drift mean sign; 0 where nothing changed | extra L/s through the link per extra L/s of k's demand |
| $N$, $N_r$ | $\lvert g_w\rvert$; $\lvert g_w\rvert_{s,k}/\lvert g_w\rvert_{s,\text{home}}$ on the same window (undefined below `own_min`) | native response; relative to the own district (size-free) |
| $S$ | $\mathrm{rms}_h[z(PQ^{(k)}_s)-z(PQ^{(0)}_s)]$ | how far the weekly shape a client sees moves, in sd |
| $\sigma$ | $\lvert g_{ie0}\rvert/(\lvert g_{de0}\rvert+\lvert g_{ie0}\rvert)$, $g_{de0}$ = gain of $B(D^{(k)},\text{traj}_0)-B(D^{(0)},\text{traj}_0)$; undefined below `sigma_floor` | share of the response that needs the hub to react |
| interaction | $\lvert g_{de1}-g_{de0}\rvert/(\lvert g_{de0}\rvert+\lvert g_{ie0}\rvert)$, $g_{de1}$ = gain of $B(D^{(k)},\text{traj}_k)-B(D^{(0)},\text{traj}_k)$ | large: the split depends on which course the hub is held on (not unique) |
| $S_{de0}$ | S of $B(D^{(k)},\text{traj}_0)$ vs $B(D^{(0)},\text{traj}_0)$ | shape effect of demand transit alone |
| $m_k$, $\Delta m_k$ | k's share of the link's district-bound water, no-drift city (baseline window); its change under k's drift (settled window) | carrying k's water; how the drift changes it |
| channel | none if the label is below its floor; else demand ($\sigma$ < `ch_lo`), both, storage ($\sigma$ > `ch_hi`), undefined | which mechanism carries a response |
| $\gamma_S$, S·dose | `dose_response`'s dose exponent of S; S counted only where $\gamma_S\ge$ `GAMMA_MIN` | does the shape label grow with the drift |

Floors: `taxonomy.S_floor`, `taxonomy.Nr_floor` (parameters.yml). Dropped from `epanet_gt_V4`: C, RE, r²ₜ, systematic
share, and every comparison against the old estimator (study gain split, study significance, switch fit).

In [1]:
import warnings, logging, sys, pathlib, json, os, shutil, pickle, time
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
from IPython.display import display, Markdown

REPO = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "src" / "fedwater").is_dir()), None)
assert REPO is not None, "run this from inside the fedwater repo"
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
from fedwater.config import load_base_params
from fedwater.experiments.spec import expand_study, load_studies
from fedwater.placement import exact as ex
from fedwater.placement import select as sel

warnings.filterwarnings("ignore")
logging.getLogger("kedro").setLevel(logging.WARNING)
pd.set_option("display.width", 230, "display.max_columns", 60, "display.max_rows", 300)

STUDIES = ["dtown_districting_placement"]   # one built world per partition: its stacks.yml names the LABEL stack
WORLDS = None                   # or {(network, partition): [sim_hash, ...]} to bypass the study expansion
STACKS = None                   # or [probe stack dir, ...]: added to WORLDS' stacks (no placed sensors); alone, the
                                # studies are skipped
FORCE_EXACT = False             # recompute the exact layer even when cached

BASE = load_base_params(REPO)
EXP_ROOT = REPO / load_studies(REPO).get("root", "data/09_experiments")
OUT = REPO / "notebooks" / "outputs" / "epanet_gt_exact"                  # everything gt_analysis reads
DOSE_CACHE = REPO / "notebooks" / "outputs" / "dose_response" / "cache"   # dose_response.ipynb's runs (optional)
OUT.mkdir(parents=True, exist_ok=True)
SP = BASE["sensor_placement"]
EXACT_CFG = SP.get("exact") or {}
RULE = {**sel.RULE_DEFAULTS, **(SP.get("selection") or {})}
TAX = {**sel.TAXONOMY_DEFAULTS, **(SP.get("taxonomy") or {})}
FLOOR = {"Nr": float(TAX["Nr_floor"]), "S": float(TAX["S_floor"])}
FLOOR["S·dose"] = FLOOR["S"]
GAMMA_MIN = 0.5                 # S·dose: the shape label counted only where gamma_S >= this (dose_response definition)
M_FLOOR = 0.05                  # composition: a district "uses" a link's water at this share or more
SWEEP = {"Nr": np.logspace(-3, 0.5, 15), "S": np.logspace(-2, 0.3, 13)}
ACC = float((BASE.get("hydraulics") or {}).get("accuracy") or 1e-5)
RES_REL = 10 * ACC              # dose_response's resolution rule for gamma
print(f"experiments {EXP_ROOT}  ->  {OUT}")

experiments c:\Users\arthu\USPy\10_Mestrado\fedWater\data\09_experiments  ->  c:\Users\arthu\USPy\10_Mestrado\fedWater\notebooks\outputs\epanet_gt_exact


In [2]:
# the probe stacks: the LABEL stack of one built world per (network, partition) -- the stack the placed sensors' labels
# come from -- or the STACKS given; their exact layers (computed here if not cached: K + 2 + 3K runs per stack)
TARGETS = []
for p in map(Path, STACKS or []):
    mf = json.loads((p / "manifest.json").read_text())
    TARGETS.append({"key": (mf["network"], mf["method"]), "stack": p, "placed": None, "world": None})
if WORLDS is not None or not STACKS:
    if WORLDS is None:
        WORLDS = {}
        for st in STUDIES:
            for w in expand_study(st, REPO)["worlds"]:
                WORLDS.setdefault((w["network"], w["flat"]["districting"]), []).append(w["sim_hash"])
    for key, hashes in WORLDS.items():
        for h in sorted(set(hashes)):
            mf = json.loads((EXP_ROOT / "worlds" / h / "manifest.json").read_text())
            if mf["status"] != "ok":
                continue
            clone = EXP_ROOT / "worlds" / h / "clone"
            ptr = yaml.safe_load((clone / "data/08_reporting/sensor_placement/stacks.yml").read_text())
            pl = pd.read_csv(clone / "data/03_primary/sensor_placement.csv", dtype={"element": str})
            TARGETS.append({"key": key, "stack": Path(ptr["label"]["path"]), "placed": pl, "world": h})
            break
for t in TARGETS:
    t["X"] = ex.ensure_exact(t["stack"], EXACT_CFG, force=FORCE_EXACT)
    mf = t["X"]["manifest"]
    for need in ("e2_traj", "periods", "cards"):
        if not mf.get(need):
            raise ValueError(f"{t['key']}: exact layer {t['X']['path'].name} was computed without {need!r} "
                             "(exact.DEFAULTS / sensor_placement.exact); recompute with FORCE_EXACT = True")
TG = pd.DataFrame([{"network": t["key"][0], "partition": t["key"][1], "stack": t["stack"].name,
                    "exact": t["X"]["path"].name, "world (placement)": t["world"],
                    "gates": f"{int(t['X']['gates']['passed'].sum())}/{len(t['X']['gates'])}",
                    "districts": len(t["X"]["manifest"]["districts"])} for t in TARGETS])
display(TG)

exact layer 1c0918c6dc18\exact\e8dcb0db4133: cached
exact layer f8d3ae7d6e79\exact\76da6583e12e: cached


,network,partition,stack,exact,world (placement),gates,districts
0,dtown,fast_greedy,1c0918c6dc18,e8dcb0db4133,0ed323104afb,14/14,5
1,dtown,girvan_newman,f8d3ae7d6e79,76da6583e12e,6b4c52748c9b,14/14,5


## Tables

Per stack, then the gates of every stack in one table.

In [3]:
# per stack: the exact tables restricted to the placement candidates, the placement, the dose exponents (if
# dose_response ran on this exact layer), the composition of every world, geometry, and the card series
def _link(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        dst.unlink()
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def dose_of(t):
    """Dose exponents per (element, k) from dose_response's cache for THIS exact layer (same definition)."""
    d = DOSE_CACHE / t["key"][0] / t["key"][1] / t["X"]["path"].name
    fs = sorted(d.glob("*__*.parquet")) if d.exists() else []
    if not fs:
        return None, d
    D = pd.concat([pd.read_parquet(f) for f in fs], ignore_index=True)
    D["element"] = D["element"].astype(str)
    lams = sorted(D["lam"].unique())
    if 1.0 not in lams or len(lams) < 2:
        return None, d
    A = D.pivot_table(index=["element", "k"], columns="lam", values="A")
    Sx = D.pivot_table(index=["element", "k"], columns="lam", values="S")
    q0 = D[D["lam"] == 1.0].set_index(["element", "k"])["mean_abs_q0"].reindex(A.index)
    lx = np.log(np.array(lams)); xc = lx - lx.mean()

    def slope(M):
        Y = np.log(M[lams].where(M[lams] > 0).to_numpy(float))
        ok = np.isfinite(Y).all(axis=1)
        out = np.full(len(M), np.nan)
        out[ok] = (Y[ok] - Y[ok].mean(axis=1, keepdims=True)) @ xc / (xc @ xc)
        return out
    res = (A[lams[0]] >= RES_REL * q0).to_numpy()
    G = pd.DataFrame({"gamma_A": np.where(res, slope(A), np.nan), "gamma_S": np.where(res, slope(Sx), np.nan),
                      "rho_S": np.where(res, Sx[lams[0]] / (lams[0] * Sx[1.0]), np.nan), "dose_resolved": res},
                     index=A.index).reset_index()
    return G, d


def channel_of(df, metric):
    """none below the label's floor; else by the storage share sigma: demand < ch_lo <= both <= ch_hi < storage;
    undefined where sigma is (masked below the gain floor, or no E2)."""
    resp = df[metric] >= FLOOR[metric]
    sh = df["sigma"]
    return np.select([~resp, sh < TAX["ch_lo"], sh > TAX["ch_hi"], sh.notna()], ["none", "demand", "storage", "both"],
                     "undefined")


T = {k: [] for k in ("links", "labels", "monthly", "pumps_monthly", "composition", "periods", "gates", "geometry",
                     "nodes", "cards")}
for t in TARGETS:
    X, (net, part) = t["X"], t["key"]
    meta = {"network": net, "partition": part}
    names = list(X["manifest"]["districts"])
    sd, sm = int(X["manifest"]["steps_day"]), int(X["manifest"]["steps_month"])
    PS = sel.prep(X, TAX)
    live = set(sel.candidates(PS, RULE).index)
    L = X["links"].assign(element=lambda d: d["element"].astype(str))
    L["candidate"] = L["element"].isin(live)
    pl = t["placed"]
    if pl is not None:
        p_ = pl.drop_duplicates("element").set_index("element")
        L["placed"] = L["element"].isin(p_.index)
        for c in ("slot", "slot_class", "filled_as", "district"):
            L[c] = L["element"].map(p_[c]) if c in p_ else None
        L = L.rename(columns={"district": "client"})
    else:
        L["placed"], L["slot"], L["slot_class"], L["filled_as"], L["client"] = False, None, None, None, None
    T["links"].append(L.assign(**meta))
    C = L[L["candidate"]]
    info = C.set_index("element")[["home", "topo", "placed", "slot_class", "mean_abs_flow"] +
                                  [f"m_{d}" for d in names]]

    # labels: every period; the final window's exact quantities repeated on each row of a (link, k)
    B = X["labels"].assign(element=lambda d: d["element"].astype(str))
    B = B[B["element"].isin(live)].copy()
    B["cls_S"], B["cls_Nr"] = sel.taxonomy(B, TAX, "S"), sel.taxonomy(B, TAX, "Nr")
    B["channel_S"], B["channel_Nr"] = channel_of(B, "S"), channel_of(B, "Nr")
    B["m_k"] = [info.at[e, f"m_{k}"] for e, k in zip(B["element"], B["k"])]       # k's share, no-drift, baseline
    G, dose_dir = dose_of(t)
    if G is not None:
        B = B.merge(G, on=["element", "k"], how="left")
    else:
        B["gamma_A"] = B["gamma_S"] = B["rho_S"] = np.nan
        B["dose_resolved"] = False
    # S counted only where it grows with the drift; undefined where no dose run exists for this stack
    B["S·dose"] = B["S"].where(B["gamma_S"] >= GAMMA_MIN, 0.0) if G is not None else np.nan
    fin_cols = [c for c in ("dm_k", "m_k", "m_k_nodrift", "m_k_drift", "c_k_drift", "dc_storage", "sigma", "interaction",
                            "g_tot", "g_de0", "g_ie0", "g_de1", "g_ie1", "S_de0", "carries", "cls_S", "cls_Nr",
                            "channel_S", "channel_Nr", "gamma_A", "gamma_S", "rho_S", "dose_resolved", "S·dose",
                            "mean_abs_dq", "amp") if c in B.columns]
    LP = X["labels_period"].assign(element=lambda d: d["element"].astype(str))
    LP = LP[LP["element"].isin(live)].drop(columns=["mean_abs_dq"])
    LB = LP.merge(B[["element", "k", "own"] + fin_cols], on=["element", "k", "own"], how="left")
    LB["N"] = LB["g_w"].abs()
    for c in ("home", "topo", "placed", "slot_class", "mean_abs_flow"):
        LB[c] = LB["element"].map(info[c])
    T["labels"].append(LB.assign(**meta))

    # month by month (candidates), relative native response to the link's own final response; pumps' on-fraction
    LM = X["labels_monthly"].assign(element=lambda d: d["element"].astype(str))
    own_fin = B[B["own"]].set_index("element")["g_w"]
    Mo = LM[LM["element"].isin(live)].copy()
    Mo["own_g_final"] = Mo["element"].map(own_fin)
    Mo["Nr_final"] = Mo["g_w"].abs() / Mo["own_g_final"].abs().where(Mo["own_g_final"].abs() >= ex.DEFAULTS["own_min"])
    Mo = Mo.merge(B[["element", "k", "channel_S", "channel_Nr"]], on=["element", "k"], how="left")
    for c in ("home", "topo", "placed"):
        Mo[c] = Mo["element"].map(info[c])
    T["monthly"].append(Mo.drop(columns=[c for c in ("on_nodrift", "on_drift") if c in Mo]).assign(**meta))
    pumps = set(L.loc[L["is_pump"].astype(bool), "element"])
    if "on_drift" in LM:
        T["pumps_monthly"].append(LM[LM["element"].isin(pumps)][["element", "k", "month", "on_nodrift", "on_drift",
                                                                   "dD_mean", "dD_abs_mean"]].assign(**meta))

    # windows (hours) per drifting district
    for k in names:
        w = X["manifest"]["windows"][k]
        T["periods"].append({**meta, "k": k, "init_lo": w["baseline"][0], "init_hi": w["baseline"][1],
                             "final_lo": w["settled"][0], "final_hi": w["settled"][1], "steps_day": sd,
                             "steps_month": sm, "hours": int(X["manifest"]["hours"])})

    # composition of every candidate link: the no-drift city (baseline window), and on each k's settled window the
    # no-drift city and the drift world (from the card's hour-of-week totals); raw shares, the bar sums to 1
    dest = names + list(ex.GROUPS_TAIL)
    base = L[L["candidate"]][["element"] + [f"c_{d}" for d in dest] + ["volume"]].rename(
        columns={f"c_{d}": d for d in dest})
    T["composition"].append(base.assign(world="nodrift", k=None, window="baseline", **meta))
    cdir = OUT / "cards" / net / part
    for k in names:
        card = ex.load_card(X["path"], k)
        _link(X["path"] / "cards" / f"{k}.npz", cdir / f"{k}.npz")
        links_c = [str(x) for x in card["links"]]
        keep = [i for i, e in enumerate(links_c) if e in live]
        for tag in ("nodrift", "drift"):
            Cw = card[f"how__{tag}"][keep].astype(float).sum(axis=1)                  # L x G window totals
            vol = Cw.sum(axis=1)
            sh = Cw / np.where(vol > 0, vol, np.nan)[:, None]
            fr = pd.DataFrame(sh, columns=list(card["dests"]))
            fr.insert(0, "element", [links_c[i] for i in keep])
            fr["volume"] = vol / max(int(card["window"][1] - card["window"][0]), 1)
            T["composition"].append(fr.assign(world=tag, k=k, window="settled", **meta))
        T["cards"].append({**meta, "k": k, "kind": "exact", "file": str((cdir / f"{k}.npz").relative_to(OUT))})
    if dose_dir.exists():
        for f in sorted(dose_dir.glob("*__card.npz")):
            _link(f, cdir / "dose" / f.name)
            k, lam = f.name.split("__")[:2]
            T["cards"].append({**meta, "k": k, "kind": "dose", "lam": float(lam),
                               "file": str((cdir / "dose" / f.name).relative_to(OUT))})

    # gates: the exact layer's, and the dose runs' (if any)
    T["gates"].append(X["gates"].assign(source="exact", **meta))
    if dose_dir.exists():
        dg = [json.loads(f.read_text()) for f in sorted(dose_dir.glob("*.json"))]
        if dg:
            DG = pd.DataFrame(dg)
            T["gates"].append(pd.DataFrame([
                {"gate": "G-D0", "check": "dose runs reached the horizon, converged", "source": "dose",
                 "value": f"{int((~DG['completed'] | (DG['unbalanced_steps'] > 0)).sum())} / {len(DG)} failing",
                 "passed": bool((DG["completed"] & (DG["unbalanced_steps"] == 0)).all())},
                {"gate": "G-D1", "check": "dosed flows = no-drift city before the drift", "source": "dose",
                 "value": f"{DG['pre_drift_max_rel'].max():.1e}", "passed": bool(DG["pre_drift_max_rel"].max() <= 1e-6)},
                {"gate": "G-D2", "check": "lambda 1 re-run = the stored probe world", "source": "dose",
                 "value": f"{DG['full_vs_probe_max_rel'].max():.1e}",
                 "passed": bool(DG["full_vs_probe_max_rel"].max() <= 1e-6)}]).assign(**meta))

    # geometry for the maps: every link (pumps / valves / dead drawn grey), nodes with their district
    src = ex._physics_owner(t["stack"])
    with open(src / "worlds" / names[0] / "wn.pkl", "rb") as fh:
        wn = pickle.load(fh)
    dy = yaml.safe_load((t["stack"] / "districts.yml").read_text())
    n2d = {str(n): d for d, ns in dy["districts"].items() for n in ns}
    T["nodes"].append(pd.DataFrame([{"node": n, "x": wn.get_node(n).coordinates[0], "y": wn.get_node(n).coordinates[1],
                                     "kind": ("tank" if n in wn.tank_name_list else "reservoir"
                                              if n in wn.reservoir_name_list else "junction"),
                                     "district": n2d.get(str(n))} for n in wn.node_name_list]).assign(**meta))
    T["geometry"].append(L[["element", "start", "end", "link_type", "topo", "home", "is_pump", "is_valve", "dead",
                            "candidate", "placed"]].assign(**meta))
    print(f"{net}/{part}: {len(live)} candidate links, {len(names)} drifting districts, "
          f"dose {'yes' if G is not None else 'no'}", flush=True)

TB = {}
for name, v in T.items():
    TB[name] = (pd.concat(v, ignore_index=True) if v and isinstance(v[0], pd.DataFrame) else pd.DataFrame(v))
LBL = TB["labels"]
LF = LBL[LBL["period"] == "final"]
display(Markdown("**Gates** (exact layer; dose runs where dose_response ran on the same exact layer)"))
display(TB["gates"].pivot_table(index=["source", "gate", "check"], columns=["network", "partition"], values="value",
                                aggfunc="first"))
GATES_OK = bool(TB["gates"]["passed"].all())
display(Markdown(f"**{'All gates pass.' if GATES_OK else 'A gate failed -- read the table before anything else.'}**"))

dtown/fast_greedy: 422 candidate links, 5 drifting districts, dose yes
dtown/girvan_newman: 420 candidate links, 5 drifting districts, dose yes


**Gates** (exact layer; dose runs where dose_response ran on the same exact layer)

network                                                                          dtown                       
partition                                                                  fast_greedy          girvan_newman
source gate  check                                                                                           
dose   G-D0  dose runs reached the horizon, converged                   0 / 15 failing         0 / 15 failing
       G-D1  dosed flows = no-drift city before the drift                      0.0e+00                0.0e+00
       G-D2  lambda 1 re-run = the stored probe world                          2.4e-07                2.4e-07
exact  G-X0  every run reached the horizon, converged                   0 / 22 failing         0 / 22 failing
       G-X0b one network model across the probe worlds                      1 distinct             1 distinct
       G-X0c drift worlds re-simulated through the toolkit l...                2.2e-07                2.2e-07
       G-X1  splice exact (non-drifting demand equal across ...                0.0e+00                0.0e+00
       G-X2  probe flows = no-drift city before the drift (m...                2.2e-07                2.2e-07
       G-X3  branch response = downstream demand change (gai...                1.9e-08                2.5e-08
       G-X4a E1 destination shares of every node sum to 1                      2.2e-15                2.4e-15
       G-X4b E1 destination parts of every link sum to its |...                2.0e-15                2.4e-15
       G-X5  E1 branch composition = downstream demand shares                  0.0e+00                0.0e+00
       G-X6a twin(0, traj0) reproduces the no-drift city (ga...  5.5e-07 (1.7e-03 L/s)  7.5e-07 (3.5e-03 L/s)
       G-X6b every twin hour has a head reference                                    0                      0
       G-X6c no storage share on branches (|g_ie0|)                            1.9e-08                2.5e-08
       G-X6d twin(k, traj k) - twin(0, traj0) reproduces the...  2.8e-07 (2.7e-03 L/s)  1.0e-06 (7.2e-03 L/s)
       G-X6e every hour of every drift trajectory has a head...                      0                      0

**All gates pass.**

## Derived tables and write

In [4]:
# derived per-client and per-floor tables (final window, candidate links): client-pair matrices, floor sensitivity,
# composition power and stability, the exact channel of every response
METRICS = ["Nr", "S"] + (["S·dose"] if LF["gamma_S"].notna().any() else [])


def spear(a, b):
    a, b = pd.Series(a, dtype=float), pd.Series(b, dtype=float)
    ok = a.notna() & b.notna()
    return float(a[ok].rank().corr(b[ok].rank())) if ok.sum() > 3 else np.nan


def auc(pos, neg):
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    pos, neg = pos[np.isfinite(pos)], neg[np.isfinite(neg)]
    if not len(pos) or not len(neg):
        return np.nan
    r = pd.Series(np.concatenate([pos, neg])).rank().to_numpy()
    return float((r[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))


# A. share of client j's links whose label responds to k, and the median label; all links / placed; per topo class
MAT, MAT_T = [], []
for (n, m), g in LF.groupby(["network", "partition"]):
    full = pd.MultiIndex.from_product([sorted(g["home"].unique()), sorted(g["k"].unique())], names=["home", "k"])
    for x in METRICS:
        for scope, q_ in (("all", g), ("placed", g[g["placed"]])):
            r_ = q_.assign(r=(q_[x] >= FLOOR[x]).where(q_[x].notna())).groupby(["home", "k"]).agg(
                share=("r", "mean"), n=("r", "count"), median=(x, "median")).reindex(full).reset_index()
            MAT.append(r_.assign(network=n, partition=m, scope=scope, metric=x, floor=FLOOR[x]))
        for tp_ in ("branch", "loop", "source_bridge"):
            q_ = g[g["topo"] == tp_]
            r_ = q_.assign(r=(q_[x] >= FLOOR[x]).where(q_[x].notna())).groupby(["home", "k"]).agg(
                share=("r", "mean"), n=("r", "count"), median=(x, "median")).reindex(full).reset_index()
            MAT_T.append(r_.assign(network=n, partition=m, scope=tp_, metric=x, floor=FLOOR[x]))
TB["matrix"] = pd.concat(MAT, ignore_index=True).rename(columns={"home": "j"})
TB["matrix_topo"] = pd.concat(MAT_T, ignore_index=True).rename(columns={"home": "j"})

# S. the labels against their floor: how many respond, district-specificity, branches that cannot respond
rows = []
for (n, m), g in LF.groupby(["network", "partition"]):
    off, on = g[~g["own"]], g[g["own"]]
    for x in METRICS:
        if off[x].isna().all():
            continue                                      # e.g. S·dose on a stack without dose runs
        for eps in SWEEP["S" if x == "S·dose" else x]:
            r_off = off[x] >= eps
            per_link = off.assign(r=r_off).groupby("element").agg(n_resp=("r", "sum"), n=("r", "size"))
            anyk = per_link["n_resp"] > 0
            nocarry = (off["topo"] == "branch") & (off["carries"] == False)                # noqa: E712
            rows.append({"network": n, "partition": m, "metric": x, "eps": eps,
                         "own responding": float((on[x] >= eps).mean()),
                         "other responding (all links)": float(r_off.mean()),
                         "other responding (placed)": float(r_off[off["placed"]].mean()) if off["placed"].any() else np.nan,
                         "other responding (loops)": float(r_off[off["topo"] == "loop"].mean()),
                         "district-blind share": float((per_link["n_resp"] == per_link["n"])[anyk].mean())
                         if anyk.any() else np.nan,
                         "branch false responses": float(r_off[nocarry].mean()) if nocarry.any() else np.nan})
TB["sensitivity"] = pd.DataFrame(rows)

# L. how the labels relate (off-diagonal cells, final window)
rows = []
for (n, m), g in LF[~LF["own"]].groupby(["network", "partition"]):
    rows.append({"network": n, "partition": m, "cells": len(g), "rho(Nr,S)": spear(g["Nr"], g["S"]),
                 "rho(N,S)": spear(g["N"], g["S"]), "rho(N, link flow)": spear(g["N"], g["mean_abs_flow"]),
                 "rho(Nr, link flow)": spear(g["Nr"], g["mean_abs_flow"]),
                 **{f"{x} >= {FLOOR[x]:g}": float((g[x] >= FLOOR[x]).mean()) for x in ("Nr", "S")},
                 "median gamma_S (S responds)": float(g[g["S"] >= FLOOR["S"]]["gamma_S"].median()),
                 "rho(m_k, S)": spear(g["m_k"], g["S"]), "rho(|dm_k|, S)": spear(g["dm_k"].abs(), g["S"])})
TB["relations"] = pd.DataFrame(rows)

# M3. does carrying k's water (m_k, no-drift baseline) or its change (dm_k) tell which drifts a link depends on?
rows = []
for (n, m), g in LF[~LF["own"]].dropna(subset=["m_k"]).groupby(["network", "partition"]):
    for lab_, resp in ((f"S >= {FLOOR['S']:g}", g["S"] >= FLOOR["S"]), (f"Nr >= {FLOOR['Nr']:g}", g["Nr"] >= FLOOR["Nr"])):
        rows.append({"network": n, "partition": m, "response": lab_, "cells": len(g), "responding": int(resp.sum()),
                     "AUC m_k": auc(g[resp]["m_k"], g[~resp]["m_k"]),
                     "AUC |dm_k|": auc(g[resp]["dm_k"].abs(), g[~resp]["dm_k"].abs()),
                     f"P(responds | m_k >= {M_FLOOR})": float(resp[g["m_k"] >= M_FLOOR].mean()) if (g["m_k"] >= M_FLOOR).any() else np.nan,
                     "P(responds | m_k < 0.01)": float(resp[g["m_k"] < 0.01].mean()) if (g["m_k"] < 0.01).any() else np.nan,
                     "share of responding cells with m_k < 0.01": float((g[resp]["m_k"] < 0.01).mean()) if resp.any() else np.nan,
                     "Spearman m_k vs S": spear(g["m_k"], g["S"]), "Spearman |dm_k| vs S": spear(g["dm_k"].abs(), g["S"])})
TB["m3"] = pd.DataFrame(rows)

# M2. stability of the no-drift composition: baseline window vs each k's settled window (1/2 L1 over districts)
CP = TB["composition"]
rows = []
for (n, m), g in CP[CP["world"] == "nodrift"].groupby(["network", "partition"]):
    dists = [c for c in g.columns if str(c).startswith("District")]
    rn = lambda q: q[dists].div(q[dists].sum(axis=1).where(q[dists].sum(axis=1) > 0), axis=0)
    b_ = g[g["window"] == "baseline"].set_index("element")
    rb = rn(b_)
    for k, gk in g[g["window"] == "settled"].groupby("k"):
        rk = rn(gk.set_index("element")).reindex(rb.index)
        rows.append(pd.DataFrame({"network": n, "partition": m, "k_window": k, "element": rb.index,
                                  "L1 baseline vs settled": 0.5 * (rk - rb).abs().sum(axis=1, min_count=1).to_numpy()}))
TB["stationarity"] = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

TB["channel"] = pd.concat([LF[["network", "partition", "k", "element", "home", "own", "topo", "placed", "sigma"]].assign(
    metric=x, channel=LF[f"channel_{x}"].to_numpy()) for x in ("S", "Nr")], ignore_index=True)

for name, df in TB.items():
    df.to_parquet(OUT / f"{name}.parquet", index=False)
(OUT / "manifest.json").write_text(json.dumps({
    "written": pd.Timestamp.now().isoformat(timespec="seconds"), "targets": TG.astype(str).to_dict("records"),
    "floor": FLOOR, "gamma_min": GAMMA_MIN, "m_floor": M_FLOOR, "taxonomy": TAX, "rule": RULE,
    "exact_cfg": EXACT_CFG, "gates_ok": GATES_OK, "metrics": METRICS}, indent=1, default=str))
display(Markdown(f"**Written to** `{OUT.relative_to(REPO)}`"))
display(pd.DataFrame([{"file": f"{k}.parquet", "rows": len(v)} for k, v in TB.items()]).set_index("file").T)

**Written to** `notebooks\outputs\epanet_gt_exact`

file,links.parquet,labels.parquet,monthly.parquet,pumps_monthly.parquet,composition.parquet,periods.parquet,gates.parquet,geometry.parquet,nodes.parquet,cards.parquet,matrix.parquet,matrix_topo.parquet,sensitivity.parquet,relations.parquet,m3.parquet,stationarity.parquet,channel.parquet
rows,918,12630,298910,7810,9262,10,34,918,814,40,300,450,82,2,4,4210,8420
